# RLVR with a Custom Reward Function

## Lab 1 - Prepare GSM8K data

This lab reuses the GSM8K math dataset from Lab 3, but prepares it for an RLVR job that uses a custom Python reward function registered as a SageMaker AI Registry Evaluator.

### Install requirements

In [ ]:
%pip install -r requirements.txt

## 1. Set up SageMaker

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
project_prefix = "gsm8k-custom-reward-rlvr"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

## 2. Load GSM8K

We keep the sample count small for workshop runtime and cost. Increase `sample_count` for larger experiments.

In [ ]:
import datasets
import pandas as pd

sample_count = 180
dataset_stream = datasets.load_dataset("openai/gsm8k", "main", split="train", streaming=True)
dataset = dataset_stream.take(sample_count)
dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset_stream.features)

print(f"Dataset size: {len(dataset)} samples")
pd.DataFrame(dataset).head()

## 3. Transform to RLVR format

The custom reward function receives `reference_answer` at runtime and scores the last assistant message in the `messages` list. The training dataset therefore keeps the normal RLVR fields and also stores a stable `id` and top-level `reference_answer` for the evaluator.

In [ ]:
import json
import re
from random import randint
from tqdm import tqdm


def extract_answer(answer_text):
    match = re.search(r"####\s*(.+)", answer_text)
    return match.group(1).strip().replace(",", "") if match else ""


def prepare_custom_reward_sample(sample, index):
    ground_truth = extract_answer(sample["answer"])
    record_id = f"gsm8k-train-{index:06d}"
    return {
        "id": record_id,
        "data_source": "openai/gsm8k",
        "prompt": [
            {
                "role": "user",
                "content": f"{sample['question']} Let's think step by step and output the final answer after \"####\".",
            }
        ],
        "ability": "math",
        "reward_model": {
            "ground_truth": ground_truth,
            "style": "custom",
        },
        "reference_answer": ground_truth,
        "extra_info": {
            "answer": sample["answer"],
            "question": sample["question"],
            "index": index,
            "split": "train",
        },
    }


records = [
    prepare_custom_reward_sample(row, idx)
    for idx, row in tqdm(enumerate(dataset), total=len(dataset), desc="Converting to custom-reward RLVR format")
]

print(json.dumps(records[randint(0, len(records) - 1)], indent=2))

## 4. Split and upload to S3

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(records, test_size=0.2, random_state=42)
train_data, test_data = train_test_split(train_data, test_size=0.1, random_state=42)

input_path = f"{default_prefix + '/' if default_prefix else ''}datasets/{project_prefix}"
train_s3_path = f"s3://{bucket_name}/{input_path}/train/train_rlvr_custom_reward.jsonl"
val_s3_path = f"s3://{bucket_name}/{input_path}/val/val_rlvr_custom_reward.jsonl"
test_s3_path = f"s3://{bucket_name}/{input_path}/test/test_rlvr_custom_reward.jsonl"

os.makedirs("./data", exist_ok=True)
for name, split in [("train", train_data), ("val", val_data), ("test", test_data)]:
    with open(f"./data/{name}_rlvr_custom_reward.jsonl", "w") as f:
        for item in split:
            f.write(json.dumps(item) + "\n")

s3_client.upload_file("./data/train_rlvr_custom_reward.jsonl", bucket_name, f"{input_path}/train/train_rlvr_custom_reward.jsonl")
s3_client.upload_file("./data/val_rlvr_custom_reward.jsonl", bucket_name, f"{input_path}/val/val_rlvr_custom_reward.jsonl")
s3_client.upload_file("./data/test_rlvr_custom_reward.jsonl", bucket_name, f"{input_path}/test/test_rlvr_custom_reward.jsonl")
shutil.rmtree("./data")

print(f"Train samples: {len(train_data)} -> {train_s3_path}")
print(f"Validation samples: {len(val_data)} -> {val_s3_path}")
print(f"Test samples: {len(test_data)} -> {test_s3_path}")

## 5. Register datasets in SageMaker AI Registry

In [ ]:
from sagemaker.ai_registry.dataset import CustomizationTechnique, DataSet

dataset_train = DataSet.create(
    name=f"{project_prefix}-train",
    source=train_s3_path,
    customization_technique=CustomizationTechnique.RLVR,
    wait=True,
)
print(f"TRAINING_DATASET ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name=f"{project_prefix}-val",
    source=val_s3_path,
    customization_technique=CustomizationTechnique.RLVR,
    wait=True,
)
print(f"VALIDATION_DATASET ARN: {dataset_val.arn}")

dataset_test = DataSet.create(
    name=f"{project_prefix}-test",
    source=test_s3_path,
    customization_technique=CustomizationTechnique.RLVR,
    wait=True,
)
print(f"TEST_DATASET ARN: {dataset_test.arn}")